[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/cs377/blob/main/supplemental/LogReg_Interpret_Pair.ipynb)

# Explaining a Prediction: Which Reasons Would You Give?
**CS 377 · companion to 18_2_2 Logistic Regression Interpretability · pair activity**

**Driver:** this notebook is yours. Your partner, the **Recorder**, has the **Explaining a Prediction quiz in Canvas** open, reads each question aloud and types your agreed answers.

**How the two fit together.** The quiz is the guide; this notebook is where you do the work for the quiz questions tagged **💻 NOTEBOOK**. Each section below is named after its quiz question (**Q3** here is **Q3** in the quiz). Questions tagged **💬 NO NOTEBOOK** have no section here: just talk them through.

**For each notebook question:** the code is written for you. The Recorder reads the question aloud; you find the matching section, run the cell as given (Shift+Enter), make the one change marked **✏️** and run it again. Read the results aloud and agree with your partner on what they mean. Hints are under each section: try without them first, and open Hint 1 before Hint 2.

## Setup (run this first)
**Part 1 of the quiz (Q1-Q2) has no notebook work.** Run this cell while the Recorder reads Part 1. Its first line installs XGBoost 3.2.0, the version the quiz's numbers came from, because other versions give somewhat different results.

It rebuilds the three 18_2_2 models on the same Titanic data and the same split, and makes what every section uses:
- `df`: the full data, with the 18_2_2 features plus `sex_m` (1 = male), the opposite coding of `sex_f`.
- `X_train`, `X_test`, `y_train`, `y_test`: 665 training and 222 test passengers; in `y`, 1 means the passenger survived.
- `fit_lr(sex_column, C)`: fits the 18_2_2 sklearn pipeline (scale `age` and `fare_log`, then `LogisticRegression`) with sex coded as `sex_column`.
- `sm_model` and `or_df`: the statsmodels `Logit` fit and its odds ratios with 95% CIs (18_2_2 Section 1).
- `xgb_model`: the 18_2_2 XGBoost model.
- `PASSENGER`: the row of the test passenger used in Q4-Q6, Mr. Herman Klaber.

In [ ]:
%pip install -q xgboost==3.2.0
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression

sns.set_theme(style='whitegrid')
pd.set_option('display.width', 140)

# Data and features exactly as in 18_2_2, plus sex_m (1 = male) for Q4 and Q5
df = pd.read_csv('https://raw.githubusercontent.com/bsheese/cs377/main/data/data_titanic.csv')
df.columns = df.columns.str.lower().str.replace(' aboard', '', regex=False)
df['sibspouse']   = (df['siblings/spouses'] > 0).astype(int)
df['parentchild'] = (df['parents/children'] > 0).astype(int)
df['sex_f']       = (df['sex'] == 'female').astype(int)   # 1 = female, 0 = male
df['sex_m']       = 1 - df['sex_f']                        # 1 = male, 0 = female
df['fare_log']    = np.log1p(df['fare'])
df = pd.get_dummies(df, columns=['pclass'], prefix='pclass', drop_first=True)   # 1st class is the reference
for col in ['pclass_2', 'pclass_3']:
    df[col] = df[col].astype(int)

features = ['age', 'sibspouse', 'parentchild', 'fare_log', 'sex_f', 'pclass_2', 'pclass_3']
X = df[features]
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)


def fit_lr(sex_column='sex_f', C=1.0):
    """The 18_2_2 sklearn pipeline (scale age and fare_log, then LogisticRegression), with sex coded as sex_column.
    Returns the fitted pipeline and its feature list."""
    feats = [sex_column if f == 'sex_f' else f for f in features]
    preprocessor = ColumnTransformer([('num', StandardScaler(), ['age', 'fare_log'])], remainder='passthrough')
    pipe = Pipeline([('preprocessor', preprocessor),
                     ('classifier', LogisticRegression(C=C, random_state=42, max_iter=1000))])
    return pipe.fit(df.loc[X_train.index, feats], y_train), feats


def input_names(pipe):
    """Feature names in the order the model sees them (age and fare_log first, after scaling)."""
    return [n.split('__')[-1] for n in pipe.named_steps['preprocessor'].get_feature_names_out()]


# statsmodels Logit on the same scaled features (18_2_2 Section 1)
scaler = StandardScaler().fit(X_train[['age', 'fare_log']])
X_train_scaled = X_train.copy()
X_train_scaled[['age', 'fare_log']] = scaler.transform(X_train[['age', 'fare_log']])
sm_model = sm.Logit(y_train, sm.add_constant(X_train_scaled)).fit(disp=0)
ci = np.exp(sm_model.conf_int())
or_df = pd.DataFrame({'OR': np.exp(sm_model.params), 'OR_low': ci[0], 'OR_high': ci[1],
                      'p_value': sm_model.pvalues}).drop(index='const')

# XGBoost on the same features (18_2_2 Setup)
xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                              eval_metric='logloss', random_state=42).fit(X_train, y_train)

# The passenger for Q4-Q6 (in the test set)
PASSENGER = df.index[df['name'] == 'Mr. Herman Klaber'][0]
print(f'Train: {len(X_train)} passengers   Test: {len(X_test)} passengers')
print(df.loc[[PASSENGER], ['name', 'survived', 'sex', 'age', 'fare', 'sibspouse', 'parentchild', 'pclass_2', 'pclass_3']])

# Part 2 (Q3-Q5): Two Models, One Passenger

## Q3 · Forest plot against feature importance
> **Quiz Q3:**
>
> 18_2_2 Section 2 compares the odds ratios with XGBoost's feature importance. The Q3 cell draws a forest plot (each odds ratio with its 95% CI, and a reference line at 1) next to the importance bars, then prints both in a table. XGBoost can measure importance in more than one way. `'gain'`, the default that 18_2_2 plots, is the average improvement in the training loss per split that uses the feature. `'total_gain'` adds up the improvement over every split that uses it. The `splits` column counts those splits.
>
> - Run the cell as given. Which features have a CI that includes 1?
> - Change `IMPORTANCE_TYPE` to `'total_gain'` and run it again. Where does `fare_log` rank now (1 = most important)? Use the `splits` column to explain why its rank changed.
> - Can either importance tell you whether a higher fare raised or lowered the odds of survival? What does the forest plot say about `fare_log`?

**✏️ Your change:** Change `IMPORTANCE_TYPE = 'gain'` to `IMPORTANCE_TYPE = 'total_gain'`, then run the cell again.

**Tell your Recorder:** which CIs include 1, fare_log's rank under total_gain and why it changed, and what importance can and cannot say about fare_log.

<details><summary>Hint 1</summary>

Only the string in quotes changes. The table's `importance rank` column gives the ranks, and `splits` says how many times the trees split on each feature.

</details>

<details><summary>Hint 2</summary>

`IMPORTANCE_TYPE = 'total_gain'`

</details>

In [ ]:
IMPORTANCE_TYPE = 'gain'      # ✏️ change to 'total_gain' and run again

booster = xgb_model.get_booster()
scores = pd.Series(booster.get_score(importance_type=IMPORTANCE_TYPE)).reindex(features, fill_value=0)
splits = pd.Series(booster.get_score(importance_type='weight')).reindex(features, fill_value=0)

table = or_df[['OR', 'OR_low', 'OR_high']].copy()
table['CI includes 1'] = (table['OR_low'] < 1) & (table['OR_high'] > 1)
table['importance'] = scores / scores.sum()          # rescaled to sum to 1, as in 18_2_2
table['importance rank'] = table['importance'].rank(ascending=False).astype(int)
table['splits'] = splits.astype(int)
table = table.sort_values('OR', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
# Forest plot: each odds ratio with its 95% CI, and the reference line at OR = 1 (no effect)
axes[0].hlines(y=table.index, xmin=table['OR_low'], xmax=table['OR_high'], color='gray', lw=2)
sns.scatterplot(data=table, x='OR', y=table.index, hue='CI includes 1', palette={True: '#7f8c8d', False: '#c0392b'},
                s=90, zorder=3, ax=axes[0])
axes[0].axvline(1, color='black', linestyle='--', lw=1)
axes[0].set_xscale('log')
axes[0].set_xticks([0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10, 20], labels=['0.05', '0.1', '0.2', '0.5', '1', '2', '5', '10', '20'])
axes[0].set_title('Logistic regression: odds ratios with 95% CIs')
axes[0].set_xlabel('Odds ratio (log scale; 1 = no effect)')
axes[0].set_ylabel('')
# XGBoost importance, in the same row order
sns.barplot(x=table['importance'], y=table.index, color='#e8a838', ax=axes[1])
axes[1].set_title(f"XGBoost importance ('{IMPORTANCE_TYPE}')")
axes[1].set_xlabel('Importance (sums to 1)')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

table.round(3)

## Q4 · Explaining one passenger's prediction
> **Quiz Q4:**
>
> 18_2_2 Section 3 explains one prediction by splitting its log-odds into one contribution per feature: contribution = coefficient × model input, and log-odds = intercept + the sum of the contributions. The Q4 cell does it for one test passenger, Mr. Herman Klaber: 1st class, male, age 41, fare $26.55, traveling alone. He died. In 18_2_2's coding (`sex_f` = 1 for women) the intercept is the log-odds of a reference passenger who is male, in 1st class, traveling alone, and of average age and fare.
>
> - Run the cell as given. What is his predicted P(survive), and which features have a negative contribution?
> - What is the contribution of being male, and why is it that value?
> - Change `SEX_COLUMN` to `'sex_m'` (1 = male) and run it again. Compare the intercept, the sex coefficient, being male's contribution and his P(survive) under the two codings: which changed and which stayed the same? (Differences in the third decimal place come from the fitting algorithm and can be ignored.)
> - A lender that turns down an application must tell the applicant the main reasons. Suppose a notice names the feature with the most negative contribution as the main reason. What would it name under each coding?

**✏️ Your change:** Change `SEX_COLUMN = 'sex_f'` to `SEX_COLUMN = 'sex_m'`, then run the cell again.

**Tell your Recorder:** his P(survive) and negative contributions, being male's contribution and why, what changed and what stayed the same under sex_m, and the main reason under each coding.

<details><summary>Hint 1</summary>

The two codings are the columns `sex_f` (1 = female) and `sex_m` (1 = male). The cell refits the model with whichever one `SEX_COLUMN` names.

</details>

<details><summary>Hint 2</summary>

`SEX_COLUMN = 'sex_m'`

</details>

In [ ]:
SEX_COLUMN = 'sex_f'      # ✏️ change to 'sex_m' (1 = male) and run again

pipe, feats = fit_lr(SEX_COLUMN)
names = input_names(pipe)
coef = pipe.named_steps['classifier'].coef_[0]
intercept = pipe.named_steps['classifier'].intercept_[0]
x_raw = df.loc[[PASSENGER], names]                                  # his values
x = pipe.named_steps['preprocessor'].transform(df.loc[[PASSENGER], feats])[0]   # model input (age, fare_log scaled)

table = pd.DataFrame({'his value': x_raw.iloc[0].values, 'model input': x,
                      'coefficient': coef, 'contribution': coef * x}, index=names)
log_odds = intercept + table['contribution'].sum()
print(f'Coding: {SEX_COLUMN}')
print(f'intercept (reference passenger): {intercept:.3f}')
print(f'log-odds = intercept + contributions = {log_odds:.3f}')
print(f'P(survive) from the log-odds:  {1 / (1 + np.exp(-log_odds)):.3f}')
print(f'P(survive) from predict_proba: {pipe.predict_proba(df.loc[[PASSENGER], feats])[0, 1]:.3f}')
table.round(3) + 0.0     # + 0.0 shows -0.000 as 0.000

## Q5 · Compared with the average passenger
> **Quiz Q5:**
>
> Q4's contributions measure how Mr. Klaber differs from a reference passenger, and the coding decides who that passenger is. Another choice is to compare him with the average training passenger: contribution = coefficient × (his input − the average input), starting from the log-odds of that average passenger instead of the intercept. The Q5 cell computes his contributions under both codings, side by side. As given, `BASELINE = 'reference'` reproduces Q4.
>
> - Change `BASELINE` to `'average'` and run it. Do the two codings agree now? What P(survive) does the average passenger get?
> - Which contribution for Mr. Klaber is most negative and which is most positive? His `pclass_3` value is 0, so explain in words what a positive `pclass_3` contribution means for him.
> - Which baseline would you use for the reasons on a notice, and why?

**✏️ Your change:** Change `BASELINE = 'reference'` to `BASELINE = 'average'`, then run the cell again.

**Tell your Recorder:** whether the codings agree and the average passenger's P(survive), his most negative and most positive contributions and what the pclass_3 one means, and which baseline you would use and why.

<details><summary>Hint 1</summary>

With `'average'`, each input is compared with the mean of that input over the 665 training passengers instead of with 0. The average passenger is not a real person: it is 36.2% female, 53.8% 3rd class and 21.7% 2nd class, 31.3% traveling with a sibling or spouse, 26.0% with a parent or child, and of average age and fare.

</details>

<details><summary>Hint 2</summary>

`BASELINE = 'average'`

</details>

In [ ]:
BASELINE = 'reference'      # ✏️ change to 'average' and run again

columns, rows = {}, []
for sex_column in ['sex_f', 'sex_m']:
    pipe, feats = fit_lr(sex_column)
    pre, clf = pipe.named_steps['preprocessor'], pipe.named_steps['classifier']
    coef, intercept = clf.coef_[0], clf.intercept_[0]
    x = pre.transform(df.loc[[PASSENGER], feats])[0]
    if BASELINE == 'reference':
        base = np.zeros_like(x)                                      # the reference passenger: every input 0
    else:
        base = pre.transform(df.loc[X_train.index, feats]).mean(axis=0)   # the average training passenger
    start = intercept + coef @ base                                 # log-odds of the baseline passenger
    contrib = coef * (x - base)
    names = [n.replace(sex_column, 'sex') for n in input_names(pipe)]
    columns[f'coded as {sex_column}'] = pd.Series(contrib, index=names)
    rows.append({'coding': sex_column, 'baseline log-odds': start, 'baseline P(survive)': 1 / (1 + np.exp(-start)),
                 'his log-odds': start + contrib.sum(), 'his P(survive)': 1 / (1 + np.exp(-(start + contrib.sum())))})

print(f'Baseline: {BASELINE}')
print(pd.DataFrame(rows).set_index('coding').round(3))
print()
print("Mr. Klaber's contributions ('sex' is sex_f or sex_m):")
pd.DataFrame(columns).round(3) + 0.0     # + 0.0 shows -0.000 as 0.000

# Extra credit (Q6-Q7): What XGBoost Can Say About One Passenger
Only if you have finished Q1-Q5.

## ⭐ Q6 · XGBoost's own contributions
> **Quiz Q6:**
>
> 18_2_2 says XGBoost can approximate per-passenger contributions with post-hoc tools (SHAP). XGBoost has this built in: `predict(..., pred_contribs=True)` splits each passenger's log-odds into a starting value (`bias`, close to the model's average log-odds over the training passengers) plus one contribution per feature, so like Q5 it compares each passenger with a typical passenger rather than with a reference passenger. The Q6 cell prints Mr. Klaber's XGBoost contributions and plots every test passenger's contribution for `FEATURE` against that feature's value.
>
> - Run it as given. Not counting `bias`, what are Mr. Klaber's most positive and most negative XGBoost contributions? Compare his `fare_log` contribution with Q5's.
> - For `sex_f`, does every woman get a positive contribution and every man a negative one?
> - Change `FEATURE` to `'fare_log'` and run it again. In the plot, do passengers with about the same `fare_log` all get a contribution with the same sign? What does that say about why importance has no sign?

**✏️ Your change:** Change `FEATURE = 'sex_f'` to `FEATURE = 'fare_log'`, then run the cell again.

**Tell your Recorder:** his most positive and most negative XGBoost contributions and the fare comparison with Q5, whether sex_f's sign follows sex, and what the fare plot says about why importance has no sign.

<details><summary>Hint 1</summary>

Only the string changes; the plot and the counts follow `FEATURE`.

</details>

<details><summary>Hint 2</summary>

`FEATURE = 'fare_log'`

</details>

In [ ]:
FEATURE = 'sex_f'      # ✏️ change to 'fare_log' and run again

contribs = pd.DataFrame(xgb_model.get_booster().predict(xgb.DMatrix(X_test), pred_contribs=True),
                        columns=features + ['bias'], index=X_test.index)
p_xgb = xgb_model.predict_proba(X_test.loc[[PASSENGER]])[0, 1]
print(f"Mr. Klaber's XGBoost contributions (P(survive) = {p_xgb:.3f}):")
print(contribs.loc[PASSENGER].sort_values().round(3).to_string())
print()
c = contribs[FEATURE]
print(f'{FEATURE}: {(c > 0).sum()} test passengers with a positive contribution, {(c < 0).sum()} negative')

plot_df = pd.DataFrame({'value': X_test[FEATURE], 'contribution': c,
                        'sex': np.where(X_test['sex_f'] == 1, 'female', 'male')})
fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(data=plot_df, x='value', y='contribution', hue='sex', alpha=0.7, ax=ax)
ax.axhline(0, color='black', lw=1)
ax.set_xlabel(f'{FEATURE} (as the model sees it)')
ax.set_ylabel(f'XGBoost contribution of {FEATURE} (log-odds)')
ax.set_title(f'{FEATURE}: one point per test passenger')
plt.tight_layout()
plt.show()

## ⭐ Q7 · Does the model need fare_log?
> **Quiz Q7:**
>
> Under `'total_gain'` XGBoost ranks `fare_log` second (Q3). The Q7 cell checks how much XGBoost's cross-validated AUC depends on `fare_log` by fitting it with and without that feature. It runs 5-fold cross-validation on the 665 training passengers for XGBoost and for logistic regression and prints each fold's AUC, the mean and the standard deviation across folds.
>
> - Run it as given (all seven features) and write down both mean AUCs.
> - Put `'fare_log'` in `DROP` and run it again. In how many of the five folds is XGBoost's AUC lower without `fare_log`?
> - Is the change in XGBoost's mean AUC larger or smaller than the standard deviation across folds? What does that say about what the importance score measured?

**✏️ Your change:** Change `DROP = []` to `DROP = ['fare_log']`, then run the cell again.

**Tell your Recorder:** both mean AUCs before and after, how many folds got worse for XGBoost, the change against the SD, and what importance measured.

<details><summary>Hint 1</summary>

`DROP` is a list of feature names to leave out. The quotes go around the name inside the brackets.

</details>

<details><summary>Hint 2</summary>

`DROP = ['fare_log']`

</details>

In [ ]:
DROP = []      # ✏️ change to ['fare_log'] and run again

keep = [f for f in features if f not in DROP]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
xgb_cv = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, eval_metric='logloss', random_state=42)
lr_cv = Pipeline([
    ('preprocessor', ColumnTransformer([('num', StandardScaler(), [f for f in ['age', 'fare_log'] if f in keep])],
                                       remainder='passthrough')),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000)),
])
rows = {}
for name, model in [('XGBoost', xgb_cv), ('Logistic regression', lr_cv)]:
    s = cross_val_score(model, X_train[keep], y_train, cv=cv, scoring='roc_auc')
    rows[name] = {**{f'fold {i + 1}': v for i, v in enumerate(s)}, 'mean': s.mean(), 'SD': s.std()}
print('Features:', keep)
pd.DataFrame(rows).T.round(3)